# Engenharia de Atributos

Este notebook realiza a preparação dos dados para os modelos de Aprendizagem de Máquina.

A partir dos preços históricos dos ativos do S&P 500, serão criadas variáveis explicativas baseadas em retorno, volatilidade, tendência, volume e risco.

Também será criada a variável alvo do problema: o retorno futuro de 21 pregões.

In [24]:
import pandas as pd
import numpy as np

from pathlib import Path

## Configuração do Projeto

In [25]:
RAW_DATA_DIR = Path("../data/raw")
PROCESSED_DATA_DIR = Path("../data/processed")

PRICES_FILE = "sp500_prices_raw.csv"
FEATURES_FILE = "sp500_features.csv"

TARGET_HORIZON = 21

PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)

## Leitura dos Dados Brutos

Os dados brutos foram gerados no notebook `01_download_data.ipynb`.

Como o arquivo possui colunas hierárquicas, utilizamos duas linhas de cabeçalho.

In [26]:
prices = pd.read_csv(
    RAW_DATA_DIR / PRICES_FILE,
    header=[0, 1],
    index_col=0,
    parse_dates=True
)

prices.head()

Ticker             AZO                                              \
Price             Open        High         Low       Close  Volume   
Date                                                                 
2018-01-02  716.539978  746.539978  714.450012  736.539978  558700   
2018-01-03  740.130005  760.000000  736.760010  749.429993  597400   
2018-01-04  759.890015  769.179993  751.219971  761.260010  641000   
2018-01-05  767.380005  780.539978  764.719971  775.500000  483000   
2018-01-08  775.159973  782.429993  761.780029  766.479980  430900   

Ticker             URI                                               ...  \
Price             Open        High         Low       Close   Volume  ...   
Date                                                                 ...   
2018-01-02  166.752787  167.755704  165.576297  167.389252   813300  ...   
2018-01-03  168.228220  169.472221  167.215666  168.941833   844100  ...   
2018-01-04  169.578296  170.320823  163.580132  165.865601  1331000  ...   
2018-01-05  166.829964  167.138558  163.454791  165.267746  1078600  ...   
2018-01-08  165.180883  166.511670  163.387220  166.299515  1623600  ...   

Ticker            HAL                                                   XOM  \
Price            Open       High        Low      Close    Volume       Open   
Date                                                                          
2018-01-02  41.302977  42.079732  41.134121  41.885544   7269200  57.357491   
2018-01-03  42.045981  43.126683  41.809579  42.636990  11181900  58.274443   
2018-01-04  42.940936  43.717688  42.409029  43.591045  10142700  59.389857   
2018-01-05  43.447506  43.818999  43.016916  43.751453   8617300  59.362461   
2018-01-08  43.616366  44.207374  43.396850  44.148273   7611900  59.328262   

Ticker                                                 
Price            High        Low      Close    Volume  
Date                                                   
2018-01-02  58.301815  57.248007  58.185486  11469300  
2018-01-03  59.513013  58.041781  59.328251  13957700  
2018-01-04  59.684104  59.143511  59.410385  10863000  
2018-01-05  59.451417  58.650795  59.362461  11047600  
2018-01-08  59.636197  59.259833  59.629353  10927100  

[5 rows x 2515 columns]

## Conversão para Formato Long

O formato original possui uma coluna para cada combinação entre ativo e variável.

Para facilitar a criação das features, os dados serão convertidos para o formato long, em que cada linha representa um par `(data, ativo)`.

In [27]:
# Converte o dataframe de formato wide para long

prices_long = (
    prices
    .stack(level=0)
    .reset_index()
)

# Renomeia as colunas principais

prices_long = prices_long.rename(columns={
    "level_0": "Date",
    "Ticker": "Ticker"
})

prices_long.head()

Price,Date,Ticker,Open,High,Low,Close,Volume
0,2018-01-02,AZO,716.539978,746.539978,714.450012,736.539978,558700.0
1,2018-01-02,URI,166.752787,167.755704,165.576297,167.389252,813300.0
2,2018-01-02,KMI,11.445127,11.769989,11.363912,11.744999,16189100.0
3,2018-01-02,FDS,176.732705,178.699241,173.824044,175.278381,349100.0
4,2018-01-02,FTV,44.505808,44.621661,43.801544,44.133858,1726105.0


In [28]:
# Padronização dos nomes das colunas

prices_long.columns.name = None

prices_long = prices_long.rename(columns={
    "Date": "date",
    "Ticker": "ticker",
    "Open": "open",
    "High": "high",
    "Low": "low",
    "Close": "close",
    "Volume": "volume"
})

prices_long.head()

,date,ticker,open,high,low,close,volume
0,2018-01-02,AZO,716.539978,746.539978,714.450012,736.539978,558700.0
1,2018-01-02,URI,166.752787,167.755704,165.576297,167.389252,813300.0
2,2018-01-02,KMI,11.445127,11.769989,11.363912,11.744999,16189100.0
3,2018-01-02,FDS,176.732705,178.699241,173.824044,175.278381,349100.0
4,2018-01-02,FTV,44.505808,44.621661,43.801544,44.133858,1726105.0


## Validação Inicial

Verificamos a quantidade de observações, ativos e o período coberto pela base.

In [29]:
print(f"Quantidade de linhas: {len(prices_long):,}")
print(f"Quantidade de ativos: {prices_long['ticker'].nunique()}")
print(f"Data inicial: {prices_long['date'].min().date()}")
print(f"Data final: {prices_long['date'].max().date()}")

Quantidade de linhas: 1,067,869
Quantidade de ativos: 503
Data inicial: 2018-01-02
Data final: 2026-06-12


## Criação das Features

Nesta etapa são criadas variáveis explicativas baseadas em retorno, volatilidade, tendência, volume, risco e posição do preço.

Todas as features são calculadas usando apenas informações disponíveis até a data da observação.

In [30]:
prices_long = prices_long.sort_values(
    ["ticker", "date"]
).reset_index(drop=True)

In [31]:
#Retornos 

prices_long["ret_1d"] = prices_long.groupby("ticker")["close"].pct_change(1)
prices_long["ret_5d"] = prices_long.groupby("ticker")["close"].pct_change(5)
prices_long["ret_21d"] = prices_long.groupby("ticker")["close"].pct_change(21)

In [32]:
#Volatilidade

daily_returns = prices_long.groupby("ticker")["close"].pct_change()

prices_long["vol_21d"] = (
    daily_returns
    .groupby(prices_long["ticker"])
    .rolling(21)
    .std()
    .reset_index(level=0, drop=True)
)

prices_long["vol_63d"] = (
    daily_returns
    .groupby(prices_long["ticker"])
    .rolling(63)
    .std()
    .reset_index(level=0, drop=True)
)

In [33]:
#tendência

ma20 = prices_long.groupby("ticker")["close"].transform(
    lambda x: x.rolling(20).mean()
)

ma50 = prices_long.groupby("ticker")["close"].transform(
    lambda x: x.rolling(50).mean()
)

prices_long["sma_20_ratio"] = prices_long["close"] / ma20
prices_long["sma_50_ratio"] = prices_long["close"] / ma50

In [34]:
#volume

volume_ma21 = prices_long.groupby("ticker")["volume"].transform(
    lambda x: x.rolling(21).mean()
)

prices_long["volume_ratio_21d"] = prices_long["volume"] / volume_ma21

In [35]:
rolling_max_63 = prices_long.groupby("ticker")["close"].transform(
    lambda x: x.rolling(63).max()
)

rolling_min_21 = prices_long.groupby("ticker")["close"].transform(
    lambda x: x.rolling(21).min()
)

rolling_max_21 = prices_long.groupby("ticker")["close"].transform(
    lambda x: x.rolling(21).max()
)

prices_long["drawdown_63d"] = prices_long["close"] / rolling_max_63 - 1

denom = rolling_max_21 - rolling_min_21
prices_long["close_position_21d"] = np.where(
    denom == 0,
    np.nan,
    (prices_long["close"] - rolling_min_21) / denom
)

In [36]:

#Amplitude máxima-mínima
 
prices_long["high_low_range"] = (
    (prices_long["high"] - prices_long["low"]) /
    prices_long["close"]
)

prices_long["high_low_range_21d"] = (
    prices_long
    .groupby("ticker")["high_low_range"]
    .transform(lambda x: x.rolling(21).mean())
)

In [37]:
#target

future_close = prices_long.groupby("ticker")["close"].shift(-TARGET_HORIZON)

prices_long["target_21d"] = future_close / prices_long["close"] - 1

In [38]:
# Log-retorno do target: comprime valores extremos e deixa a distribuição
# mais simétrica, sem remover eventos reais de mercado
prices_long["target_21d_log"] = np.log(1 + prices_long["target_21d"])

## Análise Inicial das Features

Nesta etapa são avaliadas estatísticas descritivas das features criadas para identificar possíveis valores extremos ou inconsistências.

In [39]:
feature_cols = [
    "ret_1d",
    "ret_5d",
    "ret_21d",
    "vol_21d",
    "vol_63d",
    "sma_20_ratio",
    "sma_50_ratio",
    "volume_ratio_21d",
    "drawdown_63d",
    "close_position_21d",
    "high_low_range",
    "high_low_range_21d",
    "target_21d",
    "target_21d_log"
]

prices_long[feature_cols].describe()

,ret_1d,ret_5d,ret_21d,vol_21d,vol_63d,sma_20_ratio,sma_50_ratio,volume_ratio_21d,drawdown_63d,close_position_21d,high_low_range,high_low_range_21d,target_21d,target_21d_log
count,1.043925e+06,1.041913e+06,1.033873e+06,1.033853e+06,1.012727e+06,1.034859e+06,1.019769e+06,1.034356e+06,1.013230e+06,1.034356e+06,1.044429e+06,1.034356e+06,1.033873e+06,1.033873e+06
mean,6.915817e-04,3.360655e-03,1.379845e-02,1.924042e-02,1.992880e-02,1.004656e+00,1.012325e+00,1.011104e+00,-8.157017e-02,5.570620e-01,2.535467e-02,2.534415e-02,1.379845e-02,8.802861e-03
std,2.247256e-02,4.918744e-02,9.994401e-02,1.167828e-02,1.038679e-02,5.356023e-02,8.491682e-02,4.874397e-01,8.639503e-02,3.567188e-01,1.753490e-02,1.226575e-02,9.994401e-02,1.001443e-01
min,-5.386473e-01,-6.811833e-01,-8.716766e-01,1.000685e-03,1.762554e-03,1.892681e-01,1.443020e-01,0.000000e+00,-9.094134e-01,0.000000e+00,0.000000e+00,0.000000e+00,-8.716766e-01,-2.053202e+00
25%,-9.148203e-03,-2.018324e-02,-3.946746e-02,1.221316e-02,1.348963e-02,9.783032e-01,9.678535e-01,7.365442e-01,-1.175362e-01,2.232114e-01,1.501920e-02,1.803614e-02,-3.946746e-02,-4.026742e-02
50%,8.220017e-04,3.671641e-03,1.231538e-02,1.630762e-02,1.719459e-02,1.006134e+00,1.013983e+00,9.141716e-01,-5.662777e-02,6.021214e-01,2.088143e-02,2.249533e-02,1.231538e-02,1.224016e-02
75%,1.058099e-02,2.663493e-02,6.341937e-02,2.235858e-02,2.284963e-02,1.032076e+00,1.057728e+00,1.157024e+00,-1.799405e-02,9.070987e-01,3.004296e-02,2.880599e-02,6.341937e-02,6.148954e-02
max,7.459324e-01,1.192308e+00,2.184570e+00,2.246502e-01,1.498397e-01,1.953609e+00,2.581349e+00,2.099807e+01,0.000000e+00,1.000000e+00,1.212731e+00,2.624030e-01,2.184570e+00,1.158317e+00


## Investigação de Valor Extremo

Como o maior valor do target foi elevado, investigamos a observação correspondente para verificar se parece um erro de dados ou um evento real de mercado.

In [40]:
prices_long.loc[
    prices_long["target_21d"].idxmax()
]

date                  2020-03-30 00:00:00
ticker                                APA
open                             4.072004
high                             4.080559
low                              3.421852
close                            3.515953
volume                         26494600.0
ret_1d                          -0.154321
ret_5d                          -0.046404
ret_21d                         -0.835072
vol_21d                          0.174327
vol_63d                          0.110522
sma_20_ratio                      0.42321
sma_50_ratio                     0.197958
volume_ratio_21d                 1.581626
drawdown_63d                    -0.876699
close_position_21d                    0.0
high_low_range                   0.187348
high_low_range_21d               0.190184
target_21d                        2.18457
target_21d_log                   1.158317
Name: 74868, dtype: object

## Distribuição do Target

Os quantis do target são analisados para verificar a presença de outliers extremos na variável alvo.

In [41]:
prices_long["target_21d"].quantile(
    [0.001, 0.01, 0.05, 0.95, 0.99, 0.999]
)

0.001   -0.464556
0.010   -0.241707
0.050   -0.131593
0.950    0.164300
0.990    0.295331
0.999    0.594393
Name: target_21d, dtype: float64

## Limpeza da Base

Nesta etapa removemos observações com valores ausentes gerados pelas janelas móveis e pelo deslocamento do target.

In [42]:
initial_rows = len(prices_long)

prices_long = prices_long.dropna().reset_index(drop=True)

final_rows = len(prices_long)

print(f"Linhas antes da limpeza: {initial_rows:,}")
print(f"Linhas após a limpeza: {final_rows:,}")
print(f"Linhas removidas: {initial_rows - final_rows:,}")

Linhas antes da limpeza: 1,067,869
Linhas após a limpeza: 1,002,184
Linhas removidas: 65,685


## Winsorização das Features

Valores extremos nas features podem distorcer o binning interno de modelos baseados em
árvores (LightGBM, XGBoost), fazendo com que features com outliers raros dominem
artificialmente a importância do modelo.

A winsorização corta os extremos nos percentis 1% e 99%, preservando a distribuição
geral mas eliminando valores atípicos. Eventos reais de mercado são mantidos — apenas
sua magnitude é limitada ao intervalo observado na grande maioria dos casos.

In [43]:
COLS_TO_WINSORIZE = [
    "ret_1d",
    "ret_5d",
    "ret_21d",
    "vol_21d",
    "vol_63d",
    "sma_20_ratio",
    "sma_50_ratio",
    "volume_ratio_21d",
    "drawdown_63d",
    "high_low_range",
    "high_low_range_21d",
]

for col in COLS_TO_WINSORIZE:
    lower = prices_long[col].quantile(0.01)
    upper = prices_long[col].quantile(0.99)
    prices_long[col] = prices_long[col].clip(lower, upper)

print("Winsorização aplicada (percentis 1% – 99%).")
print()
print(prices_long[COLS_TO_WINSORIZE].agg(["min", "max"]).T)

Winsorização aplicada (percentis 1% – 99%).

                         min       max
ret_1d             -0.061343  0.063236
ret_5d             -0.130598  0.140888
ret_21d            -0.243144  0.293104
vol_21d             0.006645  0.067630
vol_63d             0.008172  0.059110
sma_20_ratio        0.855003  1.144518
sma_50_ratio        0.778740  1.234293
volume_ratio_21d    0.388789  2.726963
drawdown_63d       -0.391722  0.000000
high_low_range      0.007249  0.092150
high_low_range_21d  0.011230  0.074714


## Inclusão de Informações Setoriais

As informações de setor e nome da empresa serão incorporadas à base para uso posterior nos modelos e na interpretação dos clusters.

In [44]:
companies = pd.read_csv(
    RAW_DATA_DIR / "sp500_companies.csv"
)

companies = companies[
    ["Symbol", "Security", "GICS Sector"]
].rename(columns={
    "Symbol": "ticker",
    "Security": "company",
    "GICS Sector": "sector"
})

# Normaliza o formato do ticker para compatibilidade com o Yahoo Finance
# (ex: BRK.B → BRK-B)
companies["ticker"] = companies["ticker"].str.replace(".", "-", regex=False)

companies.head()

,ticker,company,sector
0,MMM,3M,Industrials
1,AOS,A. O. Smith,Industrials
2,ABT,Abbott Laboratories,Health Care
3,ABBV,AbbVie,Health Care
4,ACN,Accenture,Information Technology


In [45]:
prices_long = prices_long.merge(
    companies,
    on="ticker",
    how="left"
)

prices_long.head()

,date,ticker,open,high,low,close,volume,ret_1d,ret_5d,ret_21d,...,sma_50_ratio,volume_ratio_21d,drawdown_63d,close_position_21d,high_low_range,high_low_range_21d,target_21d,target_21d_log,company,sector
0,2018-04-04,A,60.415396,61.828070,59.944504,61.639709,5084100.0,0.000458,-0.020954,-0.028926,...,0.942796,2.021071,-0.123282,0.155921,0.030558,0.021859,0.013598,0.013507,Agilent Technologies,Health Care
1,2018-04-05,A,61.988191,62.449663,61.686821,61.856342,2120200.0,0.003515,-0.008635,-0.033525,...,0.948184,0.856200,-0.120200,0.191084,0.012332,0.021768,0.020097,0.019898,Agilent Technologies,Health Care
2,2018-04-06,A,61.225337,61.743314,59.690229,59.944511,3578600.0,-0.030908,-0.046456,-0.084762,...,0.921477,1.432671,-0.147393,0.000000,0.034250,0.021650,0.058759,0.057097,Agilent Technologies,Health Care
3,2018-04-09,A,61.140566,62.393140,60.349472,61.394852,2639800.0,0.024195,0.011795,-0.064631,...,0.946102,1.053086,-0.126764,0.210336,0.033287,0.022499,0.033441,0.032894,Agilent Technologies,Health Care
4,2018-04-10,A,62.289576,62.901737,62.063550,62.760468,1872800.0,0.022243,0.018649,-0.061032,...,0.969401,0.749793,-0.107341,0.433179,0.013355,0.022130,0.022359,0.022112,Agilent Technologies,Health Care


## Validação da Base Final

Verificamos se todas as observações possuem setor associado e se a base final mantém a estrutura esperada.

In [46]:
print(f"Quantidade de linhas: {len(prices_long):,}")
print(f"Quantidade de ativos: {prices_long['ticker'].nunique()}")
print(f"Setores ausentes: {prices_long['sector'].isna().sum()}")
print(f"Quantidade de setores: {prices_long['sector'].nunique()}")

Quantidade de linhas: 1,002,184
Quantidade de ativos: 502
Setores ausentes: 0
Quantidade de setores: 11


## Salvamento da Base Processada

A base final com features e target será armazenada em `data/processed` para ser utilizada nos próximos notebooks.

In [47]:
prices_long.to_csv(
    PROCESSED_DATA_DIR / FEATURES_FILE,
    index=False
)

print("Arquivo salvo com sucesso.")

Arquivo salvo com sucesso.
